# YOLO11 训练（Kaggle T4 版）— cascade 方案 / detect 路线

**与 `CTAI-master/CTAI_model/kaggle_yolo11_seg_train.ipynb` 无关**，那份是已废弃的 seg 路线。
本 notebook 走 **detect** 路线：YOLO 只输出 bbox，分割交给 UNet。

## 准备工作
1. 上传两个 Kaggle dataset（与 UNet notebook 共享）：
   - `ctai-code-and-splits` ← 本仓库 `ctai_model_code/`（含 splits.json、train_unet.py 等代码）
   - `ctai-yolo-dataset` ← 本仓库 `ctai_model_code/yolo_dataset/`（610 MB，含 images/labels/MANIFEST.md/rectal_tumor.yaml）
2. Notebook 选 **GPU T4 ×2**，开 internet（要装 ultralytics）
3. 默认 `SMOKE_TEST = True`：先跑 yolo11n × 3 epoch 烟雾测试。OK 后切 False 跑两轮正式训练。

## 训练计划（论文按 yolo11s 报数；yolo11n 留作消融对照）
| 阶段 | 模型 | epochs | imgsz | augment | 用途 |
|---|---|---|---|---|---|
| smoke | yolo11n | 3 | 512 | 关 mosaic/mixup | 验证管道 |
| 第一轮 | yolo11n | 50 | 512 | 默认 | 打通流程 + 消融对照 |
| 第二轮 | yolo11s | 100 | 640 | mosaic+mixup, patience=20 | 论文最终交付 |

## 输出
- `/kaggle/working/yolo_runs/` 下两组 runs（n / s）
- `/kaggle/working/yolo_output.zip`（一键打包：best.pt × 2、results.png × 2、results.csv × 2、args.yaml × 2）

In [ ]:
# === Cell 1: 环境 + 依赖 ===
import os, sys, glob, json, subprocess, shutil, hashlib
print('python:', sys.version.split()[0])
# Kaggle 已预装 torch/cv2/numpy/tqdm/albumentations/pydicom/SimpleITK，仅需装 ultralytics
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'])
from ultralytics import YOLO
import torch
print('torch :', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# === Cell 2: 定位 datasets，校验 splits fingerprint + MANIFEST sha ===
EXPECTED_FP = '84706e8b75c8f403'
EXPECTED_SPLITS_SHA16 = '0c83582ebd913eed'  # 与 yolo_dataset/MANIFEST.md 锁定值一致
KAGGLE_NS = '/kaggle/input/datasets/ramyaramyarao'

# splits.json（先试异常挂载路径，再 glob 兜底）
splits_candidates = [f'{KAGGLE_NS}/ctai-code-and-splits/ctai_model_code/splits.json'] + \
    glob.glob('/kaggle/input/**/ctai_model_code/splits.json', recursive=True)
# 去重并过滤存在文件
splits_candidates = list(dict.fromkeys(splits_candidates))
SPLITS_JSON = None
for c in splits_candidates:
    if os.path.isfile(c):
        SPLITS_JSON = c
        break
if SPLITS_JSON is None:
    raise SystemExit('找不到 splits.json，请上传 ctai-code-and-splits dataset')
with open(SPLITS_JSON, 'r', encoding='utf-8') as f:
    payload = json.load(f)
assert payload['fingerprint'] == EXPECTED_FP, \
    f"split fingerprint mismatch: {payload['fingerprint']!r}"
actual_sha = hashlib.sha256(open(SPLITS_JSON, 'rb').read()).hexdigest()[:16]
assert actual_sha == EXPECTED_SPLITS_SHA16, \
    f'splits.json sha16 mismatch: {actual_sha!r} != {EXPECTED_SPLITS_SHA16!r} (' \
    f'splits 内容若有变化，请同步更新 yolo_dataset/MANIFEST.md 并重生成数据集)'
print(f'splits OK: fp={EXPECTED_FP}, sha16={actual_sha}')

# YOLO 数据集（先试异常挂载路径，再 glob 兜底）
yaml_candidates = [f'{KAGGLE_NS}/ctai-yolo-dataset/yolo_dataset/rectal_tumor.yaml'] + \
    glob.glob('/kaggle/input/**/yolo_dataset/rectal_tumor.yaml', recursive=True)
yaml_candidates = list(dict.fromkeys(yaml_candidates))
YOLO_DS_DIR = None
for c in yaml_candidates:
    if os.path.isfile(c):
        YOLO_DS_DIR = os.path.dirname(c)
        break
if YOLO_DS_DIR is None:
    raise SystemExit('找不到 yolo_dataset/rectal_tumor.yaml，请上传 ctai-yolo-dataset')
print(f'YOLO_DS_DIR = {YOLO_DS_DIR}')
for sp in ['train', 'val', 'test']:
    img_dir = os.path.join(YOLO_DS_DIR, 'images', sp)
    lbl_dir = os.path.join(YOLO_DS_DIR, 'labels', sp)
    n_img = len(os.listdir(img_dir))
    n_lbl = len(os.listdir(lbl_dir))
    n_empty = sum(1 for f in os.listdir(lbl_dir)
                  if os.path.getsize(os.path.join(lbl_dir, f)) == 0)
    print(f'  {sp:5s}  images={n_img}  labels={n_lbl}  empty(neg)={n_empty}')

In [ ]:
# === Cell 3: 生成 Kaggle 专用 yaml（修正 path: 字段为 Kaggle 绝对路径）===
WORK = '/kaggle/working'
RUNS = os.path.join(WORK, 'yolo_runs')
os.makedirs(RUNS, exist_ok=True)

KAGGLE_YAML = os.path.join(WORK, 'rectal_tumor_kaggle.yaml')
with open(KAGGLE_YAML, 'w', encoding='utf-8') as f:
    f.write(
        '# Kaggle 注入版（由 train_yolo11_kaggle.ipynb Cell 3 生成）\n'
        f'path: {YOLO_DS_DIR}\n'
        'train: images/train\n'
        'val: images/val\n'
        'test: images/test\n\n'
        'names:\n  0: tumor\n\nnc: 1\n'
    )
print(f'KAGGLE_YAML = {KAGGLE_YAML}')
print(open(KAGGLE_YAML).read())

In [ ]:
# === Cell 4: 训练模式开关 ===
# True  → 仅跑 yolo11n × 3 epoch 烟雾，跳过两轮正式
# False → 跳过 smoke，依次跑 yolo11n×50 + yolo11s×100
SMOKE_TEST = False
print(f'SMOKE_TEST = {SMOKE_TEST}')

In [ ]:
# === Cell 5: smoke 训练（yolo11n × 3 epoch，关 augmentation） ===
if not SMOKE_TEST:
    print('[skipped] SMOKE_TEST=False，跳过烟雾测试')
else:
    print('[SMOKE_TEST] yolo11n × 3 epoch / imgsz=512 / mosaic=0 mixup=0')
    smoke_model = YOLO('yolo11n.pt')
    smoke_model.train(
        data=KAGGLE_YAML,
        epochs=3, imgsz=512, batch=16,
        device=[0, 1],
        project=RUNS, name='smoke_n', exist_ok=True,
        mosaic=0.0, mixup=0.0, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
        plots=True, save=True, verbose=True,
    )
    smoke_dir = os.path.join(RUNS, 'smoke_n')
    print('[smoke] 产物清单:')
    for root, _, files in os.walk(smoke_dir):
        for f in files:
            p = os.path.join(root, f)
            print(f'  {os.path.relpath(p, smoke_dir):30s} {os.path.getsize(p)/1e3:.1f} KB')
    print('[smoke OK] 检查上面输出：应有 weights/best.pt、results.png、results.csv、args.yaml')

In [ ]:
# === Cell 6: 第一轮正式 — yolo11n × 50 epoch（消融对照）===
if SMOKE_TEST:
    print('[skipped] SMOKE_TEST=True，跳过 yolo11n 正式训练')
else:
    print('[SMOKE_TEST is False, running full training] yolo11n × 50 epoch / imgsz=512')
    n_model = YOLO('yolo11n.pt')
    n_model.train(
        data=KAGGLE_YAML,
        epochs=50, imgsz=512, batch=16,
        device=[0, 1],
        project=RUNS, name='tumor_det_n', exist_ok=True,
        patience=15, save=True, plots=True,
        optimizer='AdamW', lr0=1e-3,
    )
    print(f'[done] best @ {RUNS}/tumor_det_n/weights/best.pt')

In [ ]:
# === Cell 7: 第二轮正式 — yolo11s × 100 epoch（论文最终交付）===
if SMOKE_TEST:
    print('[skipped] SMOKE_TEST=True，跳过 yolo11s 正式训练')
else:
    print('[SMOKE_TEST is False, running full training] yolo11s × 100 epoch / imgsz=640 / mosaic+mixup')
    s_model = YOLO('yolo11s.pt')
    s_model.train(
        data=KAGGLE_YAML,
        epochs=100, imgsz=640, batch=16,
        device=[0, 1],
        project=RUNS, name='tumor_det_s', exist_ok=True,
        patience=20, save=True, plots=True,
        optimizer='AdamW', lr0=1e-3,
        mosaic=1.0, mixup=0.15,
    )
    print(f'[done] best @ {RUNS}/tumor_det_s/weights/best.pt')

In [ ]:
# === Cell 8: 收集产物 + 打包供下载 ===
OUT_PKG = os.path.join(WORK, 'yolo_pkg')
os.makedirs(OUT_PKG, exist_ok=True)

wanted = ['weights/best.pt', 'weights/last.pt', 'results.png',
          'results.csv', 'args.yaml', 'confusion_matrix.png']

for run_name in ['smoke_n', 'tumor_det_n', 'tumor_det_s']:
    run_dir = os.path.join(RUNS, run_name)
    if not os.path.isdir(run_dir):
        print(f'[skip] {run_name} 不存在')
        continue
    dst_run = os.path.join(OUT_PKG, run_name)
    os.makedirs(dst_run, exist_ok=True)
    print(f'[pack] {run_name}:')
    for w in wanted:
        src = os.path.join(run_dir, w)
        if os.path.exists(src):
            dst_path = os.path.join(dst_run, os.path.basename(w))
            shutil.copy2(src, dst_path)
            print(f'    {w:30s} -> {dst_path} ({os.path.getsize(dst_path)/1e6:.2f} MB)')

shutil.make_archive(os.path.join(WORK, 'yolo_output'), 'zip', OUT_PKG)
size_mb = os.path.getsize(os.path.join(WORK, 'yolo_output.zip')) / 1e6
print(f'\n=> /kaggle/working/yolo_output.zip ({size_mb:.1f} MB) 已生成。下载后解压到本地：')
print('   ctai_model_code/CTAI_model/checkpoints/yolo11n_best.pt  ← from tumor_det_n/best.pt')
print('   ctai_model_code/CTAI_model/checkpoints/yolo11s_best.pt  ← from tumor_det_s/best.pt')